# Storage Access — Modern Unity Catalog Approach

**Legacy approach (deprecated):** `dbutils.fs.mount()` with credential passthrough  
**Modern approach:** Direct `abfss://` paths via OAuth2 (Service Principal stored in Key Vault)

This notebook configures Spark to authenticate to ADLS Gen2 using a Service Principal.  
Run this once per cluster session before executing other notebooks, **or** add it as the first cell in each notebook.

### Prerequisites
1. Create a Service Principal in Azure Entra ID (App Registration)
2. Assign it **Storage Blob Data Contributor** on `sadataengineeringproj` storage account
3. Store these 3 values as secrets in your Key Vault:
   - `sp-client-id`     → Application (client) ID
   - `sp-client-secret` → Client secret value
   - `sp-tenant-id`     → Directory (tenant) ID
4. Create a Databricks Secret Scope backed by that Key Vault (see cell below)

## Step 1 — Create Databricks Secret Scope (run once, in Databricks CLI or URL trick)

Open this URL in your browser (replace workspace URL):
```
https://<your-databricks-workspace-url>#secrets/createScope
```
- **Scope name:** `kv-scope`
- **Manage principal:** All Users
- **DNS Name:** your Key Vault URI (e.g. `https://kv-dataengproj.vault.azure.net/`)
- **Resource ID:** your Key Vault resource ID from Azure Portal → Key Vault → Properties

In [ ]:
# ── Preflight: verify secret scope + secrets exist before auth ──────────────
import sys

REQUIRED_SECRETS = ["sp-client-id", "sp-client-secret", "sp-tenant-id"]
KV_SCOPE_CHECK   = "kv-scope"

# 1. Check secret scope exists
scopes = [s.name for s in dbutils.secrets.listScopes()]
if KV_SCOPE_CHECK not in scopes:
    print(f"❌ Secret scope '{KV_SCOPE_CHECK}' not found.")
    print(f"   Available scopes: {scopes}")
    print("   → Complete step 7.3: create secret scope at #secrets/createScope")
    sys.exit(1)
print(f"✅ Secret scope '{KV_SCOPE_CHECK}' exists")

# 2. Check each required secret is readable
all_ok = True
for key in REQUIRED_SECRETS:
    try:
        val = dbutils.secrets.get(scope=KV_SCOPE_CHECK, key=key)
        print(f"✅ Secret '{key}' readable ({len(val)} chars)")
    except Exception as e:
        print(f"❌ Secret '{key}' NOT readable: {e}")
        all_ok = False

if not all_ok:
    print("
⚠️  Fix missing secrets in Key Vault before continuing.")
    sys.exit(1)

print("
✅ All preflight checks passed — safe to continue")

In [ ]:
# ── Step 2: Environment parameter (passed by ADF or set manually) ───────────────
# ADF notebook activity: Base parameters → storage_account = <env-specific name>
# Manual run: change the default value to match your environment:
#   dev  → sadataengineeringdev
#   uat  → sadataengineeringuat
#   prod → sadataengineeringproj

dbutils.widgets.text("storage_account", "sadataeng260524dev", "Storage Account")
dbutils.widgets.text("kv_scope",        "kv-scope",              "Key Vault Scope")

STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
KEY_VAULT_SCOPE = dbutils.widgets.get("kv_scope")

print(f"ℹ️  Storage Account : {STORAGE_ACCOUNT}")
print(f"ℹ️  KV Scope        : {KEY_VAULT_SCOPE}")

# ── Step 3: Configure OAuth2 authentication via Service Principal ──────────────
# Secrets are fetched from Key Vault via the Databricks secret scope

client_id     = dbutils.secrets.get(scope=KEY_VAULT_SCOPE, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=KEY_VAULT_SCOPE, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=KEY_VAULT_SCOPE, key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
               "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
               client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
               client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("✅ Spark OAuth2 config set — ADLS Gen2 access ready")

In [ ]:
# ── Step 3: Define path helpers (replaces /mnt/* mount points) ──────────────────
# Use these constants in all other notebooks instead of /mnt/bronze etc.

BRONZE = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"  # fix: was 'silver2' in legacy
GOLD   = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print(f"BRONZE : {BRONZE}")
print(f"SILVER : {SILVER}")
print(f"GOLD   : {GOLD}")

In [ ]:
# ── Step 4: Verify access ────────────────────────────────────────────────────────
display(dbutils.fs.ls(f"{BRONZE}/Sales/"))

## How to use in other notebooks

Instead of `/mnt/bronze/Sales/Address/Address.parquet`, use:

```python
# Option 1 — run the config notebook first, then use full paths
%run ./storagemount
df = spark.read.format('parquet').load(f"{BRONZE}/Sales/Address/Address.parquet")

# Option 2 — hardcode full abfss:// path (no mount needed at all)
df = spark.read.format('parquet').load(
    "abfss://bronze@sadataengineeringproj.dfs.core.windows.net/Sales/Address/Address.parquet"
)
```

## Option B — Unity Catalog External Volumes (even cleaner)

If you prefer the full Unity Catalog approach:

1. **Catalog UI** → External Data → Storage Credentials → Create (use Databricks-managed Managed Identity)
2. **External Data** → External Locations → Create → point to `abfss://bronze@sadataengineeringproj.dfs.core.windows.net/`
3. **Catalog UI** → `logistics_analytics_workspace` → `data_engineering` schema → Create Volume → External Volume
4. Then reference data as:
```python
df = spark.read.format('parquet').load(
    "/Volumes/logistics_analytics_workspace/data_engineering/bronze/Sales/Address/Address.parquet"
)
```
No auth config needed — Unity Catalog handles it transparently.